In [1]:
import sys
print(sys.executable)

/mnt/ext4_data/Projects/tpw/jabluszko/.venv/bin/python


In [13]:
import json
import pandas as pd

pd.set_option('display.max_columns', None)

In [3]:
input_filename = 'data/jabłuszko_dataset_500_complete.json'

In [4]:
df = pd.read_json(input_filename)
df.shape

(500, 9)

In [5]:
df.head(2)

,store_id,location,city,population,city_type,competition,shopping_malls,kpi,sales_mix
0,JBL001,"{'lng': 16.991039, 'lat': 51.085388}",Wrocław,640000,duże miasto,"{'radius_km': 1.45, 'counts': {'discount': 7, ...",[],"{'footfall': 5877, 'basket_size': 51.48, 'conv...","{'pieczywo': 11.97, 'warzywa i owoce': 14.01, ..."
1,JBL002,"{'lng': 18.564785, 'lat': 54.5465}",Gdynia,246000,duże miasto,"{'radius_km': 1.46, 'counts': {'discount': 5, ...","[{'name': 'Galeria Pestka', 'distance': 2.637}]","{'footfall': 2354, 'basket_size': 61.87, 'conv...","{'pieczywo': 7.47, 'warzywa i owoce': 12.46, '..."


In [6]:
import json

with open(input_filename, 'r', encoding='utf-8') as file:
    data = json.load(file)

# print(data)
print(type(data))
print(data[0])

<class 'list'>
{'store_id': 'JBL001', 'location': {'lng': 16.991039, 'lat': 51.085388}, 'city': 'Wrocław', 'population': 640000, 'city_type': 'duże miasto', 'competition': {'radius_km': 1.45, 'counts': {'discount': 7, 'non_discount_chain': 10, 'independent': 3, 'drugstores': 2}, 'stores': [{'name': 'Carrefour', 'type': 'non_discount_chain', 'sales_area_m2': 350, 'distance_km': 0.17}, {'name': 'Żabka', 'type': 'non_discount_chain', 'sales_area_m2': 286, 'distance_km': 0.18}, {'name': 'Dino', 'type': 'discount', 'sales_area_m2': 930, 'distance_km': 0.09}, {'name': 'Kaufland', 'type': 'discount', 'sales_area_m2': 1105, 'distance_km': 1.03}, {'name': 'Chata Polska', 'type': 'non_discount_chain', 'sales_area_m2': 277, 'distance_km': 0.87}, {'name': 'Sklep Spożywczy u Piotr', 'type': 'independent', 'sales_area_m2': 88, 'distance_km': 1.11}, {'name': 'Netto', 'type': 'discount', 'sales_area_m2': 1116, 'distance_km': 0.53}, {'name': 'Topaz', 'type': 'discount', 'sales_area_m2': 953, 'distance_

In [7]:
## tabela CITY
df_city = pd.json_normalize(data)[["city", "population", "city_type"]]
df_city = df_city.rename(columns={"city": "city_name"})
df_city = df_city.drop_duplicates(subset=["city_name"]).reset_index(drop=True)
df_city

,city_name,population,city_type
0,Wrocław,640000,duże miasto
1,Gdynia,246000,duże miasto
2,Sosnowiec,200000,duże miasto
3,Kielce,196000,duże miasto
4,Poznań,540000,duże miasto
...,...,...,...
414,Wieś71,1729,wieś
415,Wieś72,1535,wieś
416,Wieś73,1035,wieś
417,Wieś74,2502,wieś


In [8]:
df_city['city_name'].unique()

<StringArray>
[  'Wrocław',    'Gdynia', 'Sosnowiec',    'Kielce',    'Poznań',    'Lublin',
 'Bydgoszcz',      'Łódź',    'Kraków',    'Gdańsk',
 ...
    'Wieś66',    'Wieś67',    'Wieś68',    'Wieś69',    'Wieś70',    'Wieś71',
    'Wieś72',    'Wieś73',    'Wieś74',    'Wieś75']
Length: 419, dtype: str

In [10]:
## tabela STORE
df_store_raw = pd.json_normalize(data)

# Przekształcenie współrzędnych geograficznych do formatu WKT akceptowanego przez PostGIS/PostgreSQL (GEOGRAPHY)
# Składnia: 'POINT(longitude latitude)'
df_store_raw["geom_location"] = df_store_raw.apply(
    lambda row: f"POINT({row['location.lng']} {row['location.lat']})", axis=1
)

# Mapowanie i zmiana nazw kolumn pod strukturę tabeli STORE
store_columns_map = {
    "store_id": "store_id",
    "city": "city_name",
    "geom_location": "geom_location",
    "competition.radius_km": "competition_radius_km",
    "kpi.footfall": "kpi_footfall",
    "kpi.basket_size": "kpi_basket_size",
    "kpi.conversion_rate": "kpi_conversion_rate",
    "kpi.transactions": "kpi_transactions",
    "kpi.revenue": "kpi_revenue",
    "kpi.margin_rate": "kpi_margin_rate",
    "kpi.margin": "kpi_margin",
    "kpi.competition_score": "kpi_competition_score",
    "kpi.mall_attractiveness_score": "kpi_mall_attractiveness_score",
    # sekcja sales_mix (zamiana spacji na podkreślenia)
    "sales_mix.pieczywo": "sm_pieczywo",
    "sales_mix.warzywa i owoce": "sm_warzywa_i_owoce",
    "sales_mix.słodycze i słone przekąski": "sm_slodycze_i_slone_przekaski",
    "sales_mix.piwo": "sm_piwo",
    "sales_mix.alkohole mocne": "sm_alkohole_mocne",
    "sales_mix.papierosy": "sm_papierosy",
    "sales_mix.fast food": "sm_fast_food",
    "sales_mix.woda i napoje niealkoholowe": "sm_woda_i_napoje_niealkoholowe",
    "sales_mix.sery i wędliny": "sm_sery_i_wedliny",
}

df_store = df_store_raw[list(store_columns_map.keys())].rename(
    columns=store_columns_map
)

# Konwersja typów numerycznych dla precyzji
df_store["kpi_revenue"] = df_store["kpi_revenue"].round(2)
df_store["kpi_margin"] = df_store["kpi_margin"].round(2)

In [14]:
df_store.head()

,store_id,city_name,geom_location,competition_radius_km,kpi_footfall,kpi_basket_size,kpi_conversion_rate,kpi_transactions,kpi_revenue,kpi_margin_rate,kpi_margin,kpi_competition_score,kpi_mall_attractiveness_score,sm_pieczywo,sm_warzywa_i_owoce,sm_slodycze_i_slone_przekaski,sm_piwo,sm_alkohole_mocne,sm_papierosy,sm_fast_food,sm_woda_i_napoje_niealkoholowe,sm_sery_i_wedliny
0,JBL001,Wrocław,POINT(16.991039 51.085388),1.45,5877,51.48,0.202,1187,61106.76,0.091,5560.72,0.216,0.000,11.97,14.01,11.32,9.26,6.82,11.54,21.26,8.87,4.95
1,JBL002,Gdynia,POINT(18.564785 54.5465),1.46,2354,61.87,0.280,659,40772.33,0.081,3302.56,0.442,0.158,7.47,12.46,8.45,12.09,13.20,10.79,20.87,7.19,7.48
2,JBL003,Wrocław,POINT(17.07559 51.087729),2.28,6986,64.36,0.234,1635,105228.60,0.107,11259.46,0.296,0.000,8.68,19.01,15.36,6.52,7.22,11.19,17.89,10.32,3.81
3,JBL004,Sosnowiec,POINT(19.09253 50.27751),2.73,2045,41.22,0.245,501,20651.22,0.113,2333.59,0.465,0.000,7.40,7.97,10.25,8.89,8.05,15.84,20.12,8.86,12.62
4,JBL005,Kielce,POINT(20.628885 50.901372),1.31,2816,46.22,0.234,659,30458.98,0.084,2558.55,0.014,0.461,8.97,24.76,14.44,6.03,2.31,8.07,24.07,6.41,4.94


In [15]:
## tabela COMPETITION
df_comp_raw = pd.json_normalize(
    data, record_path=["competition", "stores"], meta=["store_id"]
)

# Tworzenie ID sieci (chain_id): czyszczenie nazwy, lowercase, usunięcie spacji/polskich znaków
def generate_chain_id(name):
    translation_table = str.maketrans("ąężźćółśń ", "aezzcolsn_")
    return name.lower().translate(translation_table)

df_comp_raw["chain_id"] = df_comp_raw["name"].apply(generate_chain_id)

# Generowanie tabeli słownikowej CHAIN
df_chain = df_comp_raw[["chain_id", "name", "type"]].rename(
    columns={"name": "chain_name", "type": "chain_type"}
)
df_chain = df_chain.drop_duplicates(subset=["chain_id"]).reset_index(drop=True)

# Generowanie unikalnego klucza sztucznego competitor_id (od 1 dla każdego sklepu)
df_comp_raw["competitor_id"] = (
    df_comp_raw.groupby("store_id").cumcount() + 1
)

# Budowanie docelowej tabeli COMPETITION
df_competition = df_comp_raw[
    [
        "store_id",
        "competitor_id",
        "chain_id",
        "sales_area_m2",
        "distance_km",
    ]
].rename(columns={"sales_area_m2": "competitor_sales_area"})

# Dostosowanie typów danych
df_competition["competitor_sales_area"] = df_competition[
    "competitor_sales_area"
].astype(int)
df_competition["distance_km"] = df_competition["distance_km"].round(3)

In [16]:
df_competition.head()

,store_id,competitor_id,chain_id,competitor_sales_area,distance_km
0,JBL001,1,carrefour,350,0.17
1,JBL001,2,zabka,286,0.18
2,JBL001,3,dino,930,0.09
3,JBL001,4,kaufland,1105,1.03
4,JBL001,5,chata_polska,277,0.87


In [17]:
# tabela SHOPPING MALL
# Rozwijamy listę galerii handlowych (shopping_malls)
df_mall_raw = pd.json_normalize(
    data, record_path=["shopping_malls"], meta=["store_id"]
)

if not df_mall_raw.empty:
    # Generowanie unikalnego klucza sekwencyjnego dla galerii w obrębie sklepu
    df_mall_raw["mall_id"] = df_mall_raw.groupby("store_id").cumcount() + 1
    df_shopping_mall = df_mall_raw[
        ["store_id", "mall_id", "name", "distance"]
    ].rename(columns={"name": "mall_name", "distance": "distance_km"})
    df_shopping_mall["distance_km"] = df_shopping_mall["distance_km"].round(3)
else:
    # Obsługa pustej tabeli (gdyby żaden sklep nie miał galerii w JSON)
    df_shopping_mall = pd.DataFrame(
        columns=["store_id", "mall_id", "mall_name", "distance_km"]
    )

In [20]:
# ==============================================================================
# PREZENTACJA WYNIKÓW (Weryfikacja struktur danych)
# ==============================================================================
print("--- TABELA: CITY ---")
print(df_city.head(5).to_string(), "\n")

print("--- TABELA: CHAIN (Przykładowe rekordy) ---")
print(df_chain.head(5).to_string(), "\n")

print("--- TABELA: STORE (Wybrane kolumny KPI/Spatial) ---")
print(
    df_store[
        [
            "store_id",
            "city_name",
            "geom_location",
            "kpi_revenue",
            "sm_pieczywo",
        ]
    ].tail(10).to_string(),
    "\n",
)

print(
    f"--- TABELA: COMPETITION (Liczba wierszy: {len(df_competition)}, powiązane ze STORE) ---"
)
print(df_competition.head(5).to_string(), "\n")

print("--- TABELA: SHOPPING_MALL ---")
print(df_shopping_mall.head().to_string())

--- TABELA: CITY ---
   city_name  population    city_type
0    Wrocław      640000  duże miasto
1     Gdynia      246000  duże miasto
2  Sosnowiec      200000  duże miasto
3     Kielce      196000  duże miasto
4     Poznań      540000  duże miasto 

--- TABELA: CHAIN (Przykładowe rekordy) ---
       chain_id    chain_name          chain_type
0     carrefour     Carrefour  non_discount_chain
1         zabka         Żabka  non_discount_chain
2          dino          Dino            discount
3      kaufland      Kaufland            discount
4  chata_polska  Chata Polska  non_discount_chain 

--- TABELA: STORE (Wybrane kolumny KPI/Spatial) ---
    store_id city_name               geom_location  kpi_revenue  sm_pieczywo
490   JBL491    Wieś66  POINT(17.859707 53.001961)       406.80        15.12
491   JBL492    Wieś67  POINT(20.761894 53.721774)       128.72        17.43
492   JBL493    Wieś68  POINT(17.075618 49.396139)       374.50         9.07
493   JBL494    Wieś69  POINT(15.184405 54.

In [22]:
df_store.shape

(500, 22)

In [21]:
df_city.to_csv('data/city.csv', index=False)
df_chain.to_csv('data/chain.csv', index=False)
df_store.to_csv('data/store.csv', index=False)
df_competition.to_csv('data/competition.csv', index=False)
df_shopping_mall.to_csv('data/shopping_mall.csv', index=False)